# Experiment 2 — ST-GCN Joint Motion, NTU60 XSub

This notebook runs the controlled Joint Motion experiment for 16 outer epochs with `RepeatDataset(times=5)`, independently evaluates its best checkpoint, and compares its full validation predictions against the frozen Joint baseline. It does not modify or retrain the Joint model.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/mzuyyy/Human-action-recognition.git'
PROJECT_DIR = Path('/kaggle/working/ntu-action-recognition')
MMACTION2_DIR = Path('/kaggle/working/mmaction2')
CONFIG_PATH = PROJECT_DIR / 'configs/stgcn_ntu60_xsub_joint_motion_80e.py'
WORK_DIR = PROJECT_DIR / 'work_dirs/stgcn_ntu60_xsub_joint_motion_80e'
ANN_FILE = PROJECT_DIR / 'data/skeleton/ntu60_2d.pkl'
BASELINE_EVAL_DIR = PROJECT_DIR / 'artifacts/evaluation'
BASELINE_CHECKPOINT = PROJECT_DIR / 'artifacts/checkpoints/stgcn_joint_ntu60_xsub_best.pth'
MOTION_DIR = PROJECT_DIR / 'artifacts/experiments/joint_motion'
COMPARISON_DIR = PROJECT_DIR / 'artifacts/comparison'
NTU60_URL = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl'

# Leave both as None for the first, genuinely fresh run. After a Kaggle
# interruption set both to paths from the saved Joint Motion run.
RESUME_CHECKPOINT = None
RESUME_METRICS = None

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
for directory in (WORK_DIR, BASELINE_EVAL_DIR, MOTION_DIR, COMPARISON_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_DIR)


In [ ]:
%%bash
set -euo pipefail
python -m pip uninstall -q -y mmcv mmcv-lite >/dev/null 2>&1 || true
python -m pip install -q --only-binary=mmcv-lite \
  "importlib-metadata" "mmengine>=0.7.1,<1.0.0" "mmcv-lite==2.1.0"

MMACTION2_SRC=/kaggle/working/mmaction2
if [ ! -d "${MMACTION2_SRC}/.git" ]; then
  git -c advice.detachedHead=false clone --branch v1.2.0 --depth 1 \
    https://github.com/open-mmlab/mmaction2.git "${MMACTION2_SRC}"
else
  if ! git -C "${MMACTION2_SRC}" rev-parse -q --verify \
      "refs/tags/v1.2.0^{commit}" >/dev/null; then
    git -C "${MMACTION2_SRC}" fetch -q --depth 1 origin tag v1.2.0
  fi
  git -c advice.detachedHead=false -C "${MMACTION2_SRC}" \
    checkout -q --detach v1.2.0
fi
python -m pip uninstall -q -y mmaction2 >/dev/null 2>&1 || true
python -m pip install -q -e "${MMACTION2_SRC}"

python - <<'PY'
from pathlib import Path
path = Path('/kaggle/working/mmaction2/mmaction/utils/dependency.py')
text = path.read_text()
old = "WITH_MULTIMODAL = all(\n    satisfy_requirement(item) for item in ['transformers>=4.28.0'])"
new = "# Disabled for this skeleton-only environment.\nWITH_MULTIMODAL = False"
if old in text:
    path.write_text(text.replace(old, new))
elif new not in text:
    raise RuntimeError(f'Could not disable MMAction2 multimodal imports in {path}')
PY

PYTHONPATH=/kaggle/working/mmaction2:/kaggle/working/ntu-action-recognition \
python - <<'PY'
import mmaction
import mmaction.datasets
import mmaction.models
print('mmaction ->', mmaction.__version__, mmaction.__file__)
PY


In [ ]:
# Dataset is restored independently; the Joint baseline remains read-only.
import glob
import json
import urllib.request

if not ANN_FILE.exists():
    hits = glob.glob('/kaggle/input/**/ntu60_2d.pkl', recursive=True)
    ANN_FILE.parent.mkdir(parents=True, exist_ok=True)
    if hits:
        ANN_FILE.symlink_to(Path(hits[0]).resolve())
    else:
        temporary = ANN_FILE.with_suffix('.pkl.part')
        temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(NTU60_URL, temporary)
        temporary.replace(ANN_FILE)
print('dataset:', ANN_FILE)

baseline_names = (
    'baseline_metrics.json', 'predictions.csv', 'y_true.npy',
    'y_pred.npy', 'y_score.npy')
if not all((BASELINE_EVAL_DIR / name).is_file() for name in baseline_names):
    candidate_dirs = []
    for metrics_path in Path('/kaggle/input').rglob('baseline_metrics.json'):
        if all((metrics_path.parent / name).is_file() for name in baseline_names):
            candidate_dirs.append(metrics_path.parent)
    if len(candidate_dirs) != 1:
        raise FileNotFoundError(
            'Run notebook 03 in this session or attach exactly one baseline '
            f'evaluation bundle. Candidates: {candidate_dirs}')
    for name in baseline_names:
        destination = BASELINE_EVAL_DIR / name
        if not destination.is_file():
            destination.symlink_to((candidate_dirs[0] / name).resolve())

if not BASELINE_CHECKPOINT.is_file():
    candidates = sorted(Path('/kaggle/input').rglob(
        'stgcn_joint_ntu60_xsub_best.pth'))
    if len(candidates) != 1:
        raise FileNotFoundError(
            'Frozen Joint checkpoint must be preserved; attach exactly one '
            f'copy if this is a new session. Candidates: {candidates}')
    BASELINE_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    BASELINE_CHECKPOINT.symlink_to(candidates[0].resolve())

baseline_record = PROJECT_DIR / 'artifacts/experiments/joint/baseline.json'
baseline = json.loads(baseline_record.read_text())
assert baseline['top1'] == 0.8823 and baseline['top5'] == 0.9877
print('frozen Joint checkpoint:', BASELINE_CHECKPOINT)
print('frozen Joint predictions:', BASELINE_EVAL_DIR)


In [ ]:
# Validate the resolved controlled config and choose fresh/resume mode.
import shutil
import sys
import torch
from mmengine.config import Config

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
cfg = Config.fromfile(str(CONFIG_PATH))
assert cfg.work_dir == 'work_dirs/stgcn_ntu60_xsub_joint_motion_80e'
assert cfg.train_cfg.max_epochs == 16
assert cfg.train_dataloader.batch_size == 64
assert cfg.train_dataloader.dataset.times == 5
assert cfg.param_scheduler[0].T_max == 16
assert cfg.randomness.seed == 42
assert cfg.load_from is None and cfg.resume is False
assert cfg.train_dataloader.dataset.dataset.pipeline[1].feats == ['jm']

resume_checkpoint = None
resume_metrics_path = None
last_marker = WORK_DIR / 'last_checkpoint'
if RESUME_CHECKPOINT is not None:
    external_checkpoint = Path(RESUME_CHECKPOINT)
    external_metrics = Path(RESUME_METRICS) if RESUME_METRICS else None
    if not external_checkpoint.is_file() or not (external_metrics and external_metrics.is_file()):
        raise FileNotFoundError(
            'External resume requires both a full Joint Motion checkpoint '
            'and its epoch_metrics.jsonl')
    local_metrics = WORK_DIR / 'epoch_metrics.jsonl'
    if local_metrics.exists() and local_metrics.read_bytes() != external_metrics.read_bytes():
        raise RuntimeError('local and external resume metric histories differ')
    resume_checkpoint = external_checkpoint
    resume_metrics_path = external_metrics
elif last_marker.is_file():
    marked = Path(last_marker.read_text().strip())
    resume_checkpoint = marked if marked.is_absolute() else WORK_DIR / marked.name
    if not resume_checkpoint.is_file():
        raise FileNotFoundError(f'last_checkpoint target is missing: {resume_checkpoint}')
    resume_metrics_path = WORK_DIR / 'epoch_metrics.jsonl'
    if not resume_metrics_path.is_file():
        raise FileNotFoundError('local resume checkpoint has no epoch_metrics.jsonl')
elif list(WORK_DIR.glob('*.pth')) or (WORK_DIR / 'epoch_metrics.jsonl').exists():
    raise RuntimeError(
        'Partial Joint Motion outputs exist without last_checkpoint. Set '
        'RESUME_CHECKPOINT/RESUME_METRICS explicitly or use a clean workdir.')

if resume_checkpoint:
    if resume_checkpoint.resolve() == BASELINE_CHECKPOINT.resolve():
        raise RuntimeError('the frozen Joint checkpoint cannot resume Joint Motion')
    checkpoint = torch.load(
        resume_checkpoint, map_location='cpu', weights_only=False)
    metadata = checkpoint.get('meta', {})
    checkpoint_epoch = int(metadata.get('epoch', 0))
    if not 1 <= checkpoint_epoch <= 16:
        raise RuntimeError(
            f'invalid resume checkpoint epoch: {checkpoint_epoch}')
    if not isinstance(checkpoint.get('optimizer'), dict):
        raise RuntimeError(
            'resume requires a full checkpoint with optimizer state')
    if not checkpoint.get('param_schedulers'):
        raise RuntimeError(
            'resume requires a full checkpoint with scheduler state')
    checkpoint_config = ''.join(str(metadata.get('cfg', '')).split())
    if ("feats=['jm']" not in checkpoint_config and
            'feats=["jm"]' not in checkpoint_config):
        raise RuntimeError(
            'resume checkpoint is not identified as Joint Motion')
    resume_history = [
        json.loads(line) for line in resume_metrics_path.read_text().splitlines()
        if line.strip()]
    resume_epochs = [int(row['outer_epoch']) for row in resume_history]
    if resume_epochs != list(range(1, checkpoint_epoch + 1)):
        raise RuntimeError(
            'resume metrics must contain each outer epoch through the '
            f'checkpoint exactly once; got {resume_epochs}')
    local_metrics = WORK_DIR / 'epoch_metrics.jsonl'
    if not local_metrics.exists():
        shutil.copy2(resume_metrics_path, local_metrics)

launch_mode = 'resume' if resume_checkpoint else 'fresh'
if launch_mode == 'fresh':
    assert not list(WORK_DIR.glob('*.pth'))
resolved_path = WORK_DIR / 'resolved_config.py'
resolved_path.write_text(cfg.pretty_text)
(WORK_DIR / 'launch.json').write_text(json.dumps({
    'mode': launch_mode,
    'resume_checkpoint': str(resume_checkpoint) if resume_checkpoint else None,
    'seed': 42, 'outer_epochs': 16, 'repeat_times': 5,
}, indent=2))
print('launch mode:', launch_mode)
print('resume checkpoint:', resume_checkpoint)
print('resolved config:', resolved_path)


In [ ]:
# Train only Joint Motion. A same-config resume restores optimizer/scheduler.
import time

environment = os.environ.copy()
environment['PYTHONPATH'] = (
    str(MMACTION2_DIR) + os.pathsep + str(PROJECT_DIR) + os.pathsep
    + environment.get('PYTHONPATH', ''))
environment['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
environment['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, str(MMACTION2_DIR / 'tools/train.py'),
    str(CONFIG_PATH), '--work-dir', str(WORK_DIR), '--seed', '42',
]
if resume_checkpoint:
    command.extend(['--resume', str(resume_checkpoint)])
console_path = WORK_DIR / 'training_console.log'
started = time.perf_counter()
with console_path.open('a' if resume_checkpoint else 'w') as console:
    process = subprocess.Popen(
        command, cwd=PROJECT_DIR, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        bufsize=1)
    for line in process.stdout:
        print(line, end='')
        console.write(line)
        console.flush()
    return_code = process.wait()
elapsed = time.perf_counter() - started
if return_code != 0:
    raise RuntimeError(f'Joint Motion training failed with code {return_code}')
print(f'training command completed in {elapsed / 3600:.2f} hours')


In [ ]:
# Require the exact clean 1..16 history before independent evaluation.
from collections import Counter

history_path = WORK_DIR / 'epoch_metrics.jsonl'
history = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()]
counts = Counter(int(row['outer_epoch']) for row in history)
assert counts == Counter({epoch: 1 for epoch in range(1, 17)}), counts
for row in history:
    assert row['effective_epoch'] == row['outer_epoch'] * 5, row
for epoch in range(1, 17):
    assert (WORK_DIR / f'epoch_{epoch}.pth').is_file(), epoch
assert (WORK_DIR / 'latest.pth').is_file()
assert list(WORK_DIR.glob('best_acc_top1_epoch_*.pth'))
print('| Outer | Effective | Loss | Top-1 | Top-5 | LR |')
print('|---:|---:|---:|---:|---:|---:|')
for row in sorted(history, key=lambda item: item['outer_epoch']):
    print(
        f"| {row['outer_epoch']} | {row['effective_epoch']} | "
        f"{row['train_loss']:.6f} | {row['val_acc_top1']:.4f} | "
        f"{row['val_acc_top5']:.4f} | {row['learning_rate']:.9f} |")


In [ ]:
# Freeze and independently infer the best Joint Motion checkpoint.
command = [
    sys.executable, 'scripts/evaluate_stgcn_joint.py',
    '--config', str(CONFIG_PATH), '--ann-file', str(ANN_FILE),
    '--work-dir', str(WORK_DIR),
    '--frozen-checkpoint',
    'artifacts/checkpoints/stgcn_joint_motion_ntu60_xsub_best.pth',
    '--evaluation-dir', str(MOTION_DIR),
    '--input-representation', 'joint_motion',
    '--metrics-name', 'metrics.json',
]
subprocess.run(command, cwd=PROJECT_DIR, env=environment, check=True)


In [ ]:
# Compare raw Joint and Joint Motion predictions and write the conclusion.
command = [
    sys.executable, 'scripts/analyze_joint_vs_motion.py',
    '--joint-dir', str(BASELINE_EVAL_DIR),
    '--motion-dir', str(MOTION_DIR),
    '--comparison-dir', str(COMPARISON_DIR),
    '--work-dir', str(WORK_DIR),
]
subprocess.run(command, cwd=PROJECT_DIR, env=environment, check=True)


In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((COMPARISON_DIR / 'joint_vs_joint_motion.md').read_text()))
display(Image(filename=str(MOTION_DIR / 'training_curve.png')))
display(Image(filename=str(COMPARISON_DIR / 'confusion_joint_vs_motion.png')))
print('Joint Motion and comparison artifacts:')
for directory in (MOTION_DIR, COMPARISON_DIR):
    for path in sorted(directory.rglob('*')):
        if path.is_file():
            print('-', path.relative_to(PROJECT_DIR))
